# 05 – Đánh giá và so sánh 3 model trên tập test
**Người phụ trách:** Thành viên C &nbsp;|&nbsp; **Requirement:** Req 5 – Evaluate the models using appropriate performance metrics

**Mục tiêu:** Đánh giá cả 3 model trên **cùng một tập test**, **đúng 1 lần**: Accuracy, Precision, Recall, F1 (macro), confusion matrix, F1 từng lớp, learning curves, phân tích lỗi.

**Cần có trước:** 3 file `simple_cnn.keras`, `complex_cnn_best.keras`, `resnet50v2_finetuned.keras` trong `models/`; các file `*_history.csv`, `*_summary.json` trong `results/`.

**File tạo ra:** `results/test_metrics.csv`, `results/per_class_f1.csv`, `results/top_confused_pairs.csv`, `results/classification_reports/`, `figures/cm_*.png`, `figures/per_class_f1.png`, `figures/learning_curves_all.png`, `figures/misclassified_examples.png`

> Chạy lần lượt từng cell từ trên xuống (`Shift + Enter`). Ô có chữ **✍️ NHẬN XÉT** là chỗ bạn tự viết phân tích.

## 0. Thiết lập môi trường
Cell này làm 3 việc:
1. **Trên Colab:** kết nối Google Drive (để lưu model và kết quả lâu dài) và tải bộ dữ liệu EuroSAT (~95 MB) về máy Colab.
2. **Trên máy cá nhân:** dùng luôn thư mục dự án (notebook nằm trong `notebooks/`, ảnh nằm trong `data/EuroSAT_RGB/`).
3. Tạo sẵn các thư mục lưu kết quả: `data/`, `models/`, `results/`, `figures/`.

> Trên Colab nhớ bật GPU trước: **Runtime → Change runtime type → T4 GPU**.

In [ ]:
# ==============================================================================
# 0. THIẾT LẬP MÔI TRƯỜNG – luôn chạy cell này đầu tiên
# ==============================================================================
import os
import sys

IN_COLAB = "google.colab" in sys.modules                 # True nếu đang chạy trên Google Colab

if IN_COLAB:
    from google.colab import drive                        # thư viện kết nối Google Drive của Colab
    drive.mount("/content/drive")                         # hiện hộp thoại xin quyền truy cập Drive
    PROJECT_DIR = "/content/drive/MyDrive/eurosat_project"   # thư mục Drive DÙNG CHUNG của cả nhóm
    DATA_DIR = "/content/EuroSAT_RGB"                     # ảnh gốc để trên máy Colab (không lưu vào Drive)
    if not os.path.exists(DATA_DIR):                      # chỉ tải 1 lần cho mỗi phiên Colab
        os.system('wget -q -O /content/EuroSAT_RGB.zip "https://zenodo.org/records/7711810/files/EuroSAT_RGB.zip?download=1"')
        os.system("unzip -q -o /content/EuroSAT_RGB.zip -d /content/")   # giải nén -> /content/EuroSAT_RGB/
else:
    PROJECT_DIR = os.path.abspath("..")                   # thư mục gốc của dự án (cha của notebooks/)
    DATA_DIR = os.path.join(PROJECT_DIR, "data", "EuroSAT_RGB")

# Đường dẫn các file/thư mục kết quả – giống hệt cấu trúc repo GitHub
SPLITS_CSV  = os.path.join(PROJECT_DIR, "data", "splits.csv")   # cách chia train/val/test dùng chung
MODELS_DIR  = os.path.join(PROJECT_DIR, "models")               # model đã train (.keras)
RESULTS_DIR = os.path.join(PROJECT_DIR, "results")              # bảng số liệu (.csv, .json, .txt)
FIGURES_DIR = os.path.join(PROJECT_DIR, "figures")              # hình vẽ (.png)
for folder in [os.path.dirname(SPLITS_CSV), MODELS_DIR, RESULTS_DIR, FIGURES_DIR]:
    os.makedirs(folder, exist_ok=True)                    # tạo thư mục nếu chưa có

assert os.path.exists(DATA_DIR), f"Không tìm thấy ảnh tại {DATA_DIR} – kiểm tra lại bước tải/giải nén dữ liệu"
print("Ảnh gốc       :", DATA_DIR)
print("Số thư mục lớp:", len([d for d in os.listdir(DATA_DIR) if not d.startswith(".")]))
print("Lưu kết quả tại:", PROJECT_DIR)

In [ ]:
# ==============================================================================
# 1. IMPORT THƯ VIỆN VÀ HẰNG SỐ DÙNG CHUNG
# ==============================================================================
import json                                   # đọc/ghi file .json (tóm tắt kết quả train)
import time                                   # đo thời gian train
import numpy as np                            # xử lý mảng số
import pandas as pd                           # xử lý bảng dữ liệu
import matplotlib.pyplot as plt               # vẽ hình
import seaborn as sns                         # vẽ heatmap đẹp hơn
from PIL import Image                         # đọc file ảnh .jpg

import tensorflow as tf
from tensorflow.keras.models import load_model
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             classification_report, confusion_matrix)

RANDOM_STATE = 42                             # seed cố định để kết quả lặp lại được
tf.keras.utils.set_random_seed(RANDOM_STATE)  # cố định seed cho numpy, python và tensorflow

CLASS_NAMES = ["AnnualCrop", "Forest", "HerbaceousVegetation", "Highway", "Industrial",
               "Pasture", "PermanentCrop", "Residential", "River", "SeaLake"]   # 10 lớp, theo thứ tự label_id 0..9
NUM_CLASSES = len(CLASS_NAMES)                # = 10
IMG_SIZE = 64                                 # ảnh EuroSAT có kích thước 64 x 64
BATCH_SIZE = 64                               # số ảnh mỗi lần cập nhật trọng số

print("TensorFlow:", tf.__version__)
print("GPU:", tf.config.list_physical_devices("GPU") or "KHÔNG CÓ GPU – vào Runtime → Change runtime type → T4 GPU")

## 2. Nạp tập test
Chỉ cần tập test – đọc theo đúng `splits.csv`, chuẩn hóa giống lúc train.

In [ ]:
# ==============================================================================
# 2. NẠP DỮ LIỆU THEO splits.csv
# ==============================================================================
splits = pd.read_csv(SPLITS_CSV)              # bảng gồm: filepath, label, label_id, split

def load_split(split_name):
    """Đọc toàn bộ ảnh của 1 tập (train/val/test) -> (X, y)."""
    df = splits[splits["split"] == split_name]                                   # lọc các dòng của tập này
    X = np.stack([np.array(Image.open(os.path.join(DATA_DIR, fp))) for fp in df["filepath"]])   # (N, 64, 64, 3)
    y = df["label_id"].values                                                    # nhãn dạng số 0..9
    return X, y

X_test, y_test = load_split("test")
X_test = X_test.astype("float32") / 255.0        # chuẩn hóa giống lúc train
print("X_test:", X_test.shape, "| y_test:", y_test.shape)

## 3. Kiểm tra đủ 3 model

In [ ]:
# ==============================================================================
# 3. DANH SÁCH 3 MODEL CẦN ĐÁNH GIÁ
# ==============================================================================
MODELS = {   # tên hiển thị : (file model, file lịch sử train)
    "Simple CNN":  ("simple_cnn",            "simple_cnn_history"),
    "Complex CNN": ("complex_cnn_best",      "complex_cnn_best_history"),
    "ResNet50V2":  ("resnet50v2_finetuned",  "resnet50v2_full_history"),
}
for name, (model_file, history_file) in MODELS.items():
    assert os.path.exists(os.path.join(MODELS_DIR, f"{model_file}.keras")), f"Thiếu models/{model_file}.keras"
    assert os.path.exists(os.path.join(RESULTS_DIR, f"{history_file}.csv")), f"Thiếu results/{history_file}.csv"
print("Đủ 3 model ✔")

## 4. Đánh giá trên tập test (chạy đúng 1 lần)
- **Accuracy:** tỷ lệ ảnh đoán đúng.
- **Precision / Recall / F1 (macro):** tính cho từng lớp rồi lấy trung bình – mỗi lớp quan trọng như nhau.

⚠️ Sau bước này **không** quay lại train/chỉnh model nữa.

In [ ]:
# ==============================================================================
# 4. ĐÁNH GIÁ 3 MODEL
# ==============================================================================
os.makedirs(os.path.join(RESULTS_DIR, "classification_reports"), exist_ok=True)

predictions = {}                                               # lưu nhãn dự đoán của từng model
rows = []                                                      # mỗi model 1 dòng trong bảng so sánh
for name, (model_file, _) in MODELS.items():
    model = load_model(os.path.join(MODELS_DIR, f"{model_file}.keras"))   # nạp model đã train
    y_prob = model.predict(X_test, batch_size=BATCH_SIZE, verbose=0)    # xác suất 10 lớp cho mỗi ảnh
    y_pred = np.argmax(y_prob, axis=1)                                   # lớp có xác suất lớn nhất
    predictions[name] = y_pred

    rows.append({
        "model": name,
        "accuracy":        accuracy_score(y_test, y_pred),
        "macro_precision": precision_score(y_test, y_pred, average="macro", zero_division=0),
        "macro_recall":    recall_score(y_test, y_pred, average="macro", zero_division=0),
        "macro_f1":        f1_score(y_test, y_pred, average="macro", zero_division=0),
        "params":          model.count_params(),
    })
    report = classification_report(y_test, y_pred, labels=list(range(NUM_CLASSES)),
                                   target_names=CLASS_NAMES, digits=4, zero_division=0)
    with open(os.path.join(RESULTS_DIR, "classification_reports", f"{model_file}.txt"), "w") as f:
        f.write(report)                                        # lưu báo cáo chi tiết từng lớp
    print(f"===== {name} =====\n{report}")

## 5. Bảng so sánh tổng

In [ ]:
# ==============================================================================
# 5. BẢNG SO SÁNH 3 MODEL
# ==============================================================================
def total_train_time(name):
    """Tổng thời gian train (giây) đọc từ các file *_summary.json."""
    if name == "ResNet50V2":                                  # cộng 2 pha
        files = ["resnet50v2_phase1", "resnet50v2_finetuned"]
    elif name == "Complex CNN":                                # bản tốt nhất trong 3 biến thể
        variants = pd.read_csv(os.path.join(RESULTS_DIR, "complex_cnn_variants.csv"))
        files = [variants.sort_values("best_val_loss").iloc[0]["model"]]
    else:
        files = ["simple_cnn"]
    return sum(json.load(open(os.path.join(RESULTS_DIR, f"{f}_summary.json")))["train_time_sec"] for f in files)

test_metrics = pd.DataFrame(rows)
test_metrics["train_time_min"] = [total_train_time(n) / 60 for n in test_metrics["model"]]
test_metrics = test_metrics.round(4)
test_metrics.to_csv(os.path.join(RESULTS_DIR, "test_metrics.csv"), index=False)
test_metrics

**✍️ NHẬN XÉT:** _Model nào tốt nhất theo Macro F1? Chênh lệch bao nhiêu? Đánh đổi giữa độ chính xác, số tham số và thời gian train ra sao?_

## 6. Confusion matrix
Hàng = lớp thật, cột = lớp dự đoán, đã chuẩn hóa theo hàng (mỗi hàng cộng lại = 1). Đường chéo càng đậm càng tốt.

In [ ]:
# ==============================================================================
# 6. CONFUSION MATRIX CHO TỪNG MODEL
# ==============================================================================
SHORT = {"Simple CNN": "simple", "Complex CNN": "complex", "ResNet50V2": "resnet50v2"}   # tên file hình
confusion = {}
for name, y_pred in predictions.items():
    cm = confusion_matrix(y_test, y_pred, labels=list(range(NUM_CLASSES)), normalize="true")
    confusion[name] = cm
    plt.figure(figsize=(9, 7.5))
    sns.heatmap(cm, annot=True, fmt=".2f", cmap="Blues", vmin=0, vmax=1,
                xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
    plt.xlabel("Lớp dự đoán"); plt.ylabel("Lớp thật"); plt.title(f"Confusion matrix – {name}")
    plt.xticks(rotation=45, ha="right"); plt.tight_layout()
    plt.savefig(os.path.join(FIGURES_DIR, f"cm_{SHORT[name]}.png"), dpi=150)
    plt.show()

## 7. F1 từng lớp của 3 model

In [ ]:
# ==============================================================================
# 7. F1-SCORE TỪNG LỚP
# ==============================================================================
per_class_f1 = pd.DataFrame(
    {name: f1_score(y_test, y_pred, labels=list(range(NUM_CLASSES)), average=None, zero_division=0)
     for name, y_pred in predictions.items()},
    index=CLASS_NAMES).round(4)                               # hàng = lớp, cột = model
per_class_f1.to_csv(os.path.join(RESULTS_DIR, "per_class_f1.csv"))

per_class_f1.plot(kind="bar", figsize=(12, 4.5), width=0.8)
plt.ylabel("F1-score"); plt.ylim(0, 1.05); plt.title("F1 từng lớp – 3 model")
plt.xticks(rotation=45, ha="right"); plt.legend(loc="lower right"); plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, "per_class_f1.png"), dpi=150)
plt.show()
per_class_f1

## 8. Learning curves của 3 model trên cùng 1 hình

In [ ]:
# ==============================================================================
# 8. SO SÁNH LEARNING CURVES
# ==============================================================================
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
for name, (_, history_file) in MODELS.items():
    h = pd.read_csv(os.path.join(RESULTS_DIR, f"{history_file}.csv"))
    axes[0].plot(h["epoch"], h["val_loss"], label=name)
    axes[1].plot(h["epoch"], h["val_accuracy"], label=name)
axes[0].set_title("Validation loss"); axes[1].set_title("Validation accuracy")
for ax in axes:
    ax.set_xlabel("epoch"); ax.legend()
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, "learning_curves_all.png"), dpi=150)
plt.show()

## 9. Phân tích lỗi của model tốt nhất
Tìm các cặp lớp hay bị nhầm và xem ảnh bị đoán sai.

In [ ]:
# ==============================================================================
# 9. CẶP LỚP HAY NHẦM + ẢNH BỊ ĐOÁN SAI
# ==============================================================================
best_name = test_metrics.sort_values("macro_f1", ascending=False).iloc[0]["model"]   # model có Macro F1 cao nhất
cm = confusion[best_name].copy()
np.fill_diagonal(cm, 0)                                       # bỏ đường chéo (các ô đoán đúng)
pairs = [(CLASS_NAMES[i], CLASS_NAMES[j], round(cm[i, j], 4))
         for i in range(NUM_CLASSES) for j in range(NUM_CLASSES) if cm[i, j] > 0]
top_pairs = pd.DataFrame(sorted(pairs, key=lambda p: -p[2])[:5],
                         columns=["Lớp thật", "Bị đoán thành", "Tỷ lệ"])
top_pairs.to_csv(os.path.join(RESULTS_DIR, "top_confused_pairs.csv"), index=False)
print("Model tốt nhất:", best_name)

wrong = np.where(predictions[best_name] != y_test)[0][:16]    # 16 ảnh đầu tiên bị đoán sai
plt.figure(figsize=(12, 12))
for k, idx in enumerate(wrong):
    plt.subplot(4, 4, k + 1)
    plt.imshow(X_test[idx])
    plt.title(f"Thật: {CLASS_NAMES[y_test[idx]]}\nĐoán: {CLASS_NAMES[predictions[best_name][idx]]}", fontsize=9)
    plt.axis("off")
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, "misclassified_examples.png"), dpi=150)
plt.show()
top_pairs

**✍️ NHẬN XÉT:** _Cặp lớp nào bị nhầm nhiều nhất? Nhìn các ảnh bị đoán sai, vì sao model nhầm (màu sắc, kết cấu giống nhau, một ảnh có nhiều loại đất)?_

## ✅ Xong notebook 05
Các file trong `results/` và `figures/` là số liệu và hình dùng cho báo cáo.